In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
env_path = Path(r"C:\Projects\LLM_Learning\src\.env")
load_dotenv(dotenv_path=env_path)
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
api_key=os.getenv("OPENAI_API_KEY")

In [2]:
#read the pdf file and create document
from langchain.document_loaders import PyPDFLoader

# Path to your PDF file
pdf_path = "Prompt Engineering.pdf"

# Load the PDF
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages")

C:\Projects\LLM_Learning\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 7 pages


In [3]:
print(documents)

[Document(metadata={'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Prompt Engineering', 'source': 'Prompt Engineering.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='WhatisPromptEngineering?\nPromptengineeringisapracticewithinnaturallanguageprocessing(NLP)inartificial\nintelligence,wheretextisusedtodescribethetasktheAIshouldperform.Guidedby\nthisinput,theAIgeneratesanoutput,whichcouldtakevariousforms.Thegoalisto\nusehuman-understandabletexttointeractconversationallywithmodels,allowingfor\nflexibilityinthemodel’sperformanceduetothetaskdescriptionembeddedinthe\nprompt.\nWhatarePrompts?\nPromptsaredetaileddescriptionsofthedesiredoutputfromanAImodel.They\nrepresenttheinteractionbetweentheuserandthemodelandhelpdefinewhattheAIis\nexpectedtodo.Theeffectivenessofpromptengineeringlargelydependsonhowwell\nthepromptisdesignedtoguidethemodel.\nExamplesofPromptEngineering\nPromptsinlargelanguagemodels(LLMs)likeChatGPTorGPT-3c

In [20]:
#Splitt the documents--Not vectorizing , but splitting helps to limit the tokensize
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_and_combine(doc_name: str, chunk_size: int = 1500, chunk_overlap: int = 200) -> str:
    """
    Reads a PDF, splits into chunks, and combines into a single string.
    
    Args:
        doc_name (str): Path to the PDF file
        chunk_size (int): Size of each text chunk
        chunk_overlap (int): Overlap between chunks
    
    Returns:
        str: Combined text from all chunks
    """
    loader = PyPDFLoader(doc_name)
    document = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(document)

    return "\n".join([c.page_content for c in chunks])



In [21]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers.string import StrOutputParser

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0,api_key=api_key)

prompt="""
Summarize the below document into key points.
{document}
"""
summary=ChatPromptTemplate.from_template(prompt)

summary_chain = (
        RunnablePassthrough.assign(
        #document=RunnableLambda(lambda x: chunks[0].page_content)  # pick first chunk for now
        document=RunnableLambda(lambda x: split_and_combine(x["doc_name"]))
    )
    |
    summary
    | llm
    | StrOutputParser()   # ensures clean string output
)


In [31]:
#MCQ Chain
mcq_prompt = ChatPromptTemplate.from_template(
    """You are given a summary of a document:

{summary}

Generate 5 multiple-choice questions (MCQs) based on this summary.
Each question must have 4 options (a, b, c, d) and clearly indicate the correct answer.
Format:
Example:
Q1.What is the architecture behind LLM?
a)Regression
b)Transformer
c)Encoder
d)No architecture
Ans:Transformer

"""
)

mcq_chain = (
    RunnablePassthrough.assign(
        summary=RunnableLambda(lambda x: x["summary"])  # take summary from previous chain
    )
    | mcq_prompt
    | llm
    | StrOutputParser()
)


In [32]:
#combine both

combined_chain = (
    RunnablePassthrough.assign(
        summary=summary_chain   # summary_chain is itself a Runnable, so this works
    )
    | mcq_prompt
    | llm
    | StrOutputParser()
)




In [34]:
#Run
print(combined_chain.invoke({"doc_name": "Prompt Engineering.pdf"}))

Q1. What is the primary purpose of prompt engineering in natural language processing (NLP)?
a) To create new programming languages  
b) To describe tasks for AI and guide output generation  
c) To develop hardware for AI systems  
d) To analyze user data  
Ans: b) To describe tasks for AI and guide output generation  

Q2. Which of the following is NOT an example of a prompt type mentioned in the summary?
a) Text Prompts  
b) Code Prompts  
c) Image Prompts  
d) Audio Prompts  
Ans: d) Audio Prompts  

Q3. What is a key tip for effective prompt engineering?
a) Use complex language  
b) Avoid specifying the model's role  
c) Be concise and avoid ambiguity  
d) Provide minimal context  
Ans: c) Be concise and avoid ambiguity  

Q4. What does "Few-Shot Prompting" refer to in the context of prompting techniques?
a) Performing tasks without any examples  
b) Using multiple models for a single task  
c) Providing examples to guide the model's performance  
d) Asking the model to generate ran